<a href="https://colab.research.google.com/github/LikhithR916/BIS_Lab/blob/main/CuckooSearch/1BM24CS147_TrafficSignalTimingOptimization_CuckooSearchOptimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random

# Traffic data for two phases
arrival_rate = [20, 15]       # vehicles per minute
queue_length = [30, 20]       # vehicles
waiting_time = [60, 45]        # seconds

# Traffic signal constraints
cycle_time = 120
min_green = 20
max_green = 100

# Cuckoo Search parameters
n = 10
Pa = 0.25
iterations = 100


# Fitness function
def fitness(green1):
    green1 = max(min_green, min(max_green, green1))
    green2 = cycle_time - green1

    greens = [green1, green2]

    total_wait = 0
    total_queue = 0

    for i in range(2):
        green_ratio = greens[i] / cycle_time

        # Estimate waiting time
        estimated_wait = waiting_time[i] * (1 - green_ratio)

        # Estimate queue length
        arrivals = arrival_rate[i] * cycle_time / 60
        served = 0.5 * greens[i]

        remaining_queue = max(
            0, queue_length[i] + arrivals - served
        )

        total_wait += estimated_wait
        total_queue += remaining_queue

    return total_wait + 2 * total_queue


# Initialize nests
nests = [
    random.uniform(min_green, max_green)
    for _ in range(n)
]

# Cuckoo Search
for iteration in range(iterations):

    for i in range(n):

        # Generate new solution using Levy-flight-like step
        step = random.gauss(0, 1)
        new_solution = nests[i] + step * 10

        new_solution = max(
            min_green,
            min(max_green, new_solution)
        )

        # Select random nest
        j = random.randint(0, n - 1)

        # Replace if better
        if fitness(new_solution) < fitness(nests[j]):
            nests[j] = new_solution

    # Abandon worst nests
    for _ in range(int(Pa * n)):
        worst = max(
            range(n),
            key=lambda i: fitness(nests[i])
        )

        nests[worst] = random.uniform(
            min_green, max_green
        )

# Find best solution
best_green1 = min(nests, key=fitness)
best_green2 = cycle_time - best_green1

print("Traffic Signal Optimization")
print("---------------------------")
print("North-South Green Time:",
      round(best_green1, 2), "seconds")

print("East-West Green Time:",
      round(best_green2, 2), "seconds")

print("Best Fitness:",
      round(fitness(best_green1), 2))

Traffic Signal Optimization
---------------------------
North-South Green Time: 100 seconds
East-West Green Time: 20 seconds
Best Fitness: 167.5
